# Render the Python dashboard

Open `gameplay.html` for the standalone gameplay comparison. This notebook renders the results from `../../compare_decks.ipynb`, where `user_deck_path` sets the player deck (Kenrith by default). The public references do not change. Altair and pandas are also used for the supplementary card-overlap chart.

In [ ]:
from pathlib import Path

import altair as alt
import pandas as pd

results_path = Path("../../results/deck_comparisons.csv")
comparison_rows = pd.read_csv(results_path, keep_default_na=False)
user_decks = comparison_rows["user_deck"].unique()
if len(user_decks) != 1:
    raise ValueError("Expected results for one user deck.")


In [ ]:
shared_names_chart = (
    alt.Chart(comparison_rows)
    .mark_bar(color="#237868", size=18)
    .encode(
        x=alt.X("shared_card_names:Q", title="Shared card names", scale=alt.Scale(zero=True),
                axis=alt.Axis(tickMinStep=1)),
        y=alt.Y("reference_label:N", title=None, sort="ascending",
                axis=alt.Axis(labelLimit=340, labelPadding=8)),
        href="source_url:N",
        tooltip=[alt.Tooltip("reference_deck:N", title="Reference"),
                 alt.Tooltip("reference_group:N", title="Group"),
                 alt.Tooltip("shared_card_names:Q", title="Shared names")],
    )
    .properties(width=430, height=alt.Step(34),
                title=alt.TitleParams("Shared card names", subtitle=[str(user_decks[0])], anchor="start"))
    .configure_view(stroke=None)
)
shared_names_chart.save("deck_comparison.html")


## Standalone gameplay dashboard

Render the shared results without recomputing them. The gameplay view uses no CDN or live kernel.

## Companion play guide

Map each archetype to aggro, combo, and control using qualitative teaching labels, not metagame data. Pair the pack's recognition cues and coaching questions with an illustrative play sequence and combat decision.

In [ ]:
from html import escape
from pathlib import Path
import csv
import re

pack_root = Path("../../archmages-companion")
archetype_dir = pack_root / "archetypes"
primers = {p.stem: p for p in archetype_dir.glob("[0-9][0-9]-*.md")}
library = (archetype_dir / "archetype-library.md").read_text(encoding="utf-8")
recognition = {}
for row in csv.reader(library.splitlines(), delimiter="|"):
    if len(row) == 5:
        link = re.search(r"\]\(([\w-]+)\.md\)", row[1])
        if link:
            recognition[link[1]] = row[2].strip()

with (pack_root.parent / "data/archetype_play_lines.csv").open(encoding="utf-8", newline="") as f:
    plays = list(csv.DictReader(f))
keys = [row["primer"] for row in plays]
if len(keys) != len(set(keys)) or set(keys) != primers.keys() or set(keys) != recognition.keys():
    raise ValueError("Expected one play line and recognition cue per archetype primer.")

default_primer = "03-tempo"
if default_primer not in primers:
    raise ValueError("The opening example needs a corresponding archetype primer.")
options, articles, map_rows = [], [], []
roles = ("aggro", "combo", "control")
marks = {"core": "Defining role", "build": "Depends on the build", "none": "Not emphasized here"}
stages = [("set_up", "Set up"), ("decision", "Choose your play"), ("finish", "Close the game")]
for row in plays:
    key = row["primer"]
    if None in row or not all(row.values()):
        raise ValueError(f"Check the play line for {key}.")
    text = primers[key].read_text(encoding="utf-8")
    question, = re.findall(r"^\*\*Coaching question:\*\*\s+(.+)$", text, re.MULTILINE)
    name = escape(row["archetype"])
    selected, hidden = (" selected", "") if key == default_primer else ("", " hidden")
    options.append(f'<option value="{escape(key)}"{selected}>{name}</option>')
    cells = []
    for role in roles:
        mark = row[role]
        if mark not in marks:
            raise ValueError(f"Unknown {role} mark for {key}: {mark}")
        cells.append(f'<td><i class="role-dot {mark} {role}" aria-hidden="true"></i>'
                     f'<span class="role-reader">{marks[mark]}</span></td>')
    if all(row[role] == "none" for role in roles):
        raise ValueError(f"No role shown for {key}")
    map_rows.append(f'<tr><th scope="row"><button type="button" data-choose="{escape(key)}" '
                    f'aria-controls="playLines">{name}</button></th>{"".join(cells)}</tr>')
    steps = "".join(f"<li><h4><span>0{i}</span>{title}</h4><p>{escape(row[field])}</p></li>"
                    for i, (field, title) in enumerate(stages, start=1))
    articles.append(f'<article class="pack-plan" data-primer="{escape(key)}"{hidden}><h3>{name}</h3>'
                    f'<p class="pack-recognize"><b>Look for:</b> {escape(recognition[key])}.</p>'
                    f'<p class="pack-role-note"><b>Where it fits:</b> {escape(row["role_note"])}</p>'
                    f'<ol class="play-sequence" aria-label="Play sequence">{steps}</ol>'
                    f'<div class="pack-combat"><h4>Combat decision</h4><p>{escape(row["combat_position"])}</p>'
                    f'<p><b>Your play:</b> {escape(row["combat_choice"])}</p></div>'
                    f'<p class="play-question"><strong>Agent conversation starter</strong>{escape(question)}</p></article>')
headers = "".join(f'<th scope="col">{name}</th>' for name in ("Archetype", "Aggro", "Combo", "Control"))
half = (len(map_rows) + 1) // 2
role_map = "".join(
    f'<table aria-label="Archetype roles, entries {start+1} to {min(start+half,len(map_rows))}">'
    f'<thead><tr>{headers}</tr></thead><tbody>{"".join(map_rows[start:start+half])}</tbody></table>'
    for start in range(0, len(map_rows), half)
)
pack_panel = (Path("companion-panel.html").read_text(encoding="utf-8")
              .replace("@@ROLEMAP@@", role_map)
              .replace("@@PLAYOPTIONS@@", "".join(options)).replace("@@PLAYLINES@@", "".join(articles)))


In [ ]:
import csv
from collections import defaultdict
from html import escape

def read_results(name):
    with (results_path.parent/name).open(encoding="utf-8", newline="") as source:
        rows = list(csv.DictReader(source))
    return rows

def label(text):
    return escape(str(text), quote=True)

grid_rows = read_results("gameplay_dashboard.csv")
card_rows = read_results("gameplay_cards.csv")
views = ["Mana curve","Card types","Tribes","Commander roles","Table fit"]
explanations = [
    "Nonland library cards by printed mana value. Commanders are shown separately under Commander roles.",
    "Cards in the full 100. Types overlap; optional land backs are not counted as front-face lands.",
    "Top three printed creature subtypes, including ties. Tokens and changeling abilities are not added to printed counts.",
    "What the library can do on its own, and what changes when the commander is available.",
    "Distance from the user deck in percentage points. Lower is more similar, not necessarily stronger."
]
card_examples = defaultdict(list)
for c in card_rows:
    match_keys = [("Card types",v) for v in c["types"].split("; ") if v]
    if c["optional_land_back"]=="True":
        match_keys.append(("Card types","Land backs"))
    if c["curve_bin"]:
        match_keys.append(("Mana curve",c["curve_bin"]))
    for v,k in match_keys:
        card_examples[(c["deck"],v,k)].append(f'{c["quantity"]} {c["name"]}')
seat = next(r for r in grid_rows if r["view"]=="Seating note")
def make_panel(view, index):
    by_deck = defaultdict(dict)
    fields = []
    for r in grid_rows:
        if r["view"] != view:
            continue
        if r["metric"] not in fields:
            fields.append(r["metric"])
        by_deck[(r["deck"],r["group"])][r["metric"]]=r
    maxima = {f:max([float(row[f]["value"]) for row in by_deck.values()
                     if row[f]["value"]!=""] or [1]) for f in fields}
    rows = []
    for (name,group), records in by_deck.items():
        cells = []
        for field in fields:
            r = records[field]
            content = label(r["display"])
            fraction = 0
            if r["value"] and view in ("Mana curve","Card types"):
                fraction = float(r["value"])/max(1,maxima[field])
            evidence = card_examples.get((name,view,field),[])
            title = label(", ".join(evidence))
            cells.append(f'<td title="{title}" style="--fill:{fraction*100:.1f}%">{content}</td>')
        rows.append(f'<tr data-group="{label(group)}"><th scope="row">{label(name)}<small>{label(group)}</small></th>'+''.join(cells)+'</tr>')
    header = '<th scope="col">Deck</th>'+''.join(f'<th scope="col">{label(f)}</th>' for f in fields)
    return (f'<section id="v{index}" class="panel"><h2>{label(view)}</h2><p>{explanations[index]}</p>'
            +'<div class="scroll"><table><thead><tr>'+header+'</tr></thead><tbody>'+''.join(rows)+'</tbody></table></div></section>')



In [ ]:
import json
import math
import textwrap

profiles = read_results("gameplay_profiles.csv")
by_name = {p["deck"]:p for p in profiles}
order = {name:i for i,name in enumerate(["Your deck","Bracket 4","Bracket 3","Bracket 2","Stock precon"])}
profiles.sort(key=lambda p:(order[p["group"]],p["deck"]))
by_name = {p["deck"]:p for p in profiles}
palette = ["#f8f9ee","#e1f2ba","#ace0ae","#72c9bb","#41b6c4","#2789b3","#205693","#162e67"]
def svg(width,height,items,description,attrs=""):
    return (f'<svg xmlns="http://www.w3.org/2000/svg" role="group" aria-label="{label(description)}" viewBox="0 0 {width} {height}" {attrs}>'
            + "".join(items) + "</svg>")
def txt(x,y,content,size=12,anchor="start",color="#233a35",weight=400):
    return f'<text x="{x:.2f}" y="{y:.2f}" font-size="{size}" font-weight="{weight}" fill="{color}" text-anchor="{anchor}">{label(content)}</text>'
def rect(x,y,w,h,color):
    return f'<rect x="{x:.2f}" y="{y:.2f}" width="{w:.2f}" height="{h:.2f}" fill="{color}"/>'
strategy = read_results("strategy_cells.csv")
views_matrix = list(dict.fromkeys(r["view"] for r in strategy))
matrix_notes = ["Nonland library slots by printed mana value. The commander is excluded.",
                "Distinct cards with each deployment route, including the commander.",
                "Distinct cards that can interact with each resource. A card may cover multiple targets.",
                "Distinct cards by method. Protection and restrictions are separate from removal.",
                "Distinct cards offering recursion, reuse, or a way to rebuild a board or hand.",
                "Full-deck slots by printed type. Multitype cards appear in each applicable column.",
                "Printed creature subtypes that lead at least one list (including ties). Tokens are excluded.",
                "Printed mana abilities and land-search routes. Finding a land for hand is not acceleration.",
                "Printed timing and setup. A permanent or trigger may need a later opportunity.",
                "Distinct cards, not summed modes. An instant or flash face does not make every mode instant speed.",
                "Commander color identity, not spell color. A deck can be present in several columns. Totals count decks.",
                "Share of nonland slots, including commanders. Each card counts once; multicolor and colorless are separate. Labels are slot counts.",
                "Share of colored symbols in printed nonland casting costs, including commanders. Two-color hybrid splits equally; no colored symbols means N/A. Not a mana-source test.",
                "Mutually exclusive creature-card slots. Other multitype includes land creatures and creatures with more than one additional card type.",
                "Leading subtypes from any list after omitting Human only when another printed creature subtype is present. Ties retained; tokens excluded."]
mix_colors = {"White":"#d9bd64","Blue":"#3a76a6","Black":"#383c44","Red":"#bd574d",
              "Green":"#397657","Multicolor":"#a080ac","Colorless":"#c9cfd4"}
def composition_bars(rows):
    # Keep slot counts in the labels, but give each deck equal width for comparing shares.
    width,left,right=1060,250,64
    span=width-left-right
    names=list(dict.fromkeys(r["deck"] for r in rows))
    parts=[]
    for i,tick in enumerate((0,25,50,75,100)):
        x=left+span*tick/100
        parts.append(txt(x,21,f"{tick}%",11,"middle"))
    audit=[]
    for idx,name in enumerate(names):
        deck_rows=[r for r in rows if r["deck"]==name]
        count=sum(int(r["count"]) for r in deck_rows)
        y=32+idx*27
        group=deck_rows[0]["group"].replace("Bracket ","B").replace("Stock precon","Stock").replace("Your deck","User")
        parts += [txt(0,y+15,group,10,color="#536174"),txt(38,y+15,by_name[name]["commander"] if group=="User" else name,12),
                  txt(width-right+8,y+15,f"n={count}",11,color="#536174")]
        offset=0
        for r in deck_rows:
            n=int(r["count"]);share=n/count if count else 0
            mark=f'{name} / {r["category"]}: {n} of {count} nonland slots ({share:.1%})'
            audit.append({**{k:r[k] for k in ("deck","category","count","cards")},"display":f"{n} slots, {share:.1%}"})
            if n:
                x=left+span*offset
                parts.append(f'<g role="button" tabindex="0" data-cell="{len(audit)-1}" aria-label="{label(mark)}"><title>{label(mark)}</title>'
                             +rect(x,y,max(.2,span*share-.7),21,mix_colors[r["category"]])
                             +(txt(x+span*share/2,y+15,n,12,"middle","#24323b" if r["category"] in ("Colorless","White") else "white") if share>.045 else "")+"</g>")
            offset+=share
    return svg(width,36+27*len(names),parts,"Nonland color composition",'style="min-width:760px"'),audit

def matrix(view,note,group=None):
    rows = [r for r in strategy if r["view"]==view and (not group or r["group"] in ("Your deck",group))]
    rows.sort(key=lambda r:(order[r["group"]], r["deck"], int(r["position"])))
    decks = list(dict.fromkeys(r["deck"] for r in rows))
    categories = list(dict.fromkeys(r["category"] for r in rows))
    peak = 100 if view=="Casting-cost emphasis" else max(1,max(int(r["count"] or 0) for r in rows))
    if view=="Printed card colors":
        picture,audit=composition_bars(rows)
        return {"svg":picture,"rows":audit,"note":note,"max":None,"legend":mix_colors}
    margin, top, step = 235, 64, 25
    width = max(1000,margin+len(categories)*89)
    col = (width-margin)/len(categories)
    items=[]
    for j,title in enumerate(categories):
        lines=textwrap.wrap(title, max(11,int(col/6.8)),break_long_words=False)
        for k,line in enumerate(lines):
            items.append(txt(margin+(j+.5)*col,top-9-14*(len(lines)-k-1),line,12,"middle",weight=600))
    last = None
    offsets = {}
    for i,name in enumerate(decks):
        group=by_name[name]["group"]
        if group!=last:
            offset= (offset+14 if i else 0)
            last=group
        y=top+i*step+offset
        offsets[name]=y
        group_short = {"Your deck":"USER","Stock precon":"STOCK"}.get(group,group.replace("Bracket ","B"))
        if i==0 or group!=by_name[decks[i-1]]["group"]:
            items.append(txt(0,y+16,group_short,10,color="#35635a",weight=700))
        # The full deck name is retained in the cell tooltip and card detail.
        name_short = by_name[name]["commander"] if group=="Your deck" else name
        items.append(txt(43,y+16,name_short,12,weight=600 if group=="Your deck" else 450))
    audit=[]
    for i,r in enumerate(rows):
        val=int(r["count"] or 0)
        cell_text=("N/A" if not r["count"] else f"{val}%") if view=="Casting-cost emphasis" else val
        j=int(r["position"])
        x=margin+j*col; y=offsets[r["deck"]]
        shade=round(7*val/peak)
        desc=f'{r["deck"]} / {r["category"]}: {cell_text}'
        if view=="Casting-cost emphasis":desc+=f' ({r["symbol_equivalents"]} of {r["colored_symbol_total"]} colored symbols)'
        items.append(f'<g tabindex="0" role="button" data-cell="{i}" aria-label="{label(desc)}"><title>{label(desc)}</title>'
                     +rect(x+1,y,col-2,step-2,palette[shade])
                     +txt(x+col/2,y+16,cell_text,12,"middle","#fff" if shade>=5 else "#213d3b",550)+"</g>")
        audit.append({**{k:r[k] for k in ("deck","category","count","cards")},"display":str(cell_text)})
    if view=="Commander identity":
        for j,title in enumerate(categories):
            n=sum(int(r["count"]) for r in rows if r["category"]==title)
            items.append(txt(margin+(j+.5)*col,y+step+20,f"{n} / {len(decks)} decks",12,"middle",weight=600))
        y+=26
    return {"unit":"%" if view=="Casting-cost emphasis" else "","svg":svg(width,y+step+3,items,view,f'style="min-width:{width*.77:.0f}px"'),"note":note,"max":peak,"rows":audit}
assert len(views_matrix)==len(matrix_notes), "Every comparison needs a counting definition."
matrices = {v:matrix(v,n) for v,n in zip(views_matrix,matrix_notes)}
for v,m in matrices.items():
    m["subsets"]={g:matrix(v,m["note"],g) for g in order if g!="Your deck"}


In [ ]:
chart_rows = read_results("gameplay_charts.csv")
series_colors = ["#225d94","#ac697d","#718653","#594f91"]
def profile_plot(deck,view):
    rows=[r for r in chart_rows if r["plot"]==view and (r["deck"]==deck or view=="Table fit")]
    cats=list(dict.fromkeys(r["category"] for r in rows))
    series=list(dict.fromkeys(r["series"] for r in rows))
    values={(r["category"],r["series"]):float(r["quantity"]) for r in rows}
    peak=max(values.values(),default=1) or 1
    marks=[]
    def mark(c,contents,tip,active):
        attrs = f'role="button" tabindex="0" data-metric="{label(c)}" aria-label="{label(tip)}"' if active else ""
        return f'<g {attrs}><title>{label(tip)}</title>{contents}</g>'
    if view in ("Leading tribes","Table fit"):
        width,left=440,100
        for i,c in enumerate(cats):
            v=values[c,series[0]]; y=i*27+10; x=left+265*v/peak
            marks.append(mark(c,txt(left-8,y+13,c,13,"end")+rect(left,y,x-left,17,series_colors[min(i+1,3)] if view=="Table fit" else "#437caf")+txt(x+5,y+13,f"{v:g}"),f"{c}: {v:g}",view!="Table fit"))
        return svg(width,max(60,30*len(cats)+12),marks,view)
    width,height,left,top,bottom=620,210,29,17,167
    span=(width-left-14)/len(cats)
    ceil=max(5,math.ceil(peak/5)*5)
    ypos=lambda v:bottom-(bottom-top)*v/ceil
    for tick in range(0,ceil+1,max(1,ceil//5)):
        y=ypos(tick);marks += [rect(left,y,width-left-14,.8,"#d9e4df"),txt(left-5,y+4,tick,10,"end")]
    for j,c in enumerate(cats):
        x=left+(j+.5)*span
        marks.append(txt(x,bottom+18,c,11,"middle"))
    for k,s in enumerate(series):
        color=series_colors[k]
        points=[]
        for j,c in enumerate(cats):
            x=left+(j+.5)*span;v=values[c,s];y=ypos(v)
            points.append((x,y))
        if view=="Mana curve":
            dash=["","7 4","2 3","8 3 2 3"][k]
            path=" ".join(f"{x:.1f},{y:.1f}" for x,y in points)
            marks.append(f'<polyline fill="none" stroke="{color}" stroke-width="{3 if not k else 2}" stroke-dasharray="{dash}" points="{path}"/>')
        for j,(x,y) in enumerate(points):
            c=cats[j];v=values[c,s]
            if view=="Mana curve":
                shape=f'<circle cx="{x}" cy="{y}" r="{4 if not k else 2.2}" fill="{color}"/>'
            else:
                w=span*.18;shape=rect(x-2*w+k*w,y,max(1,w-1),bottom-y,color)
            marks.append(mark(c,shape,f"{c} / {s}: {v:g}",not k))
    return svg(width,height,marks,view)
with (results_path.parent.parent/'data/reference_decks.csv').open(encoding='utf-8-sig', newline='') as source:
    urls = {r['reference_deck']:r['source_url'] for r in csv.DictReader(source)}
audits=read_results("strategy_card_evidence.csv")
route_by_name={r["deck"]:r for r in read_results("deck_routes.csv")}
role_definitions=json.loads((results_path.parent.parent/"data/capability_definitions.json").read_text(encoding="utf-8"))
payload={}
for p in profiles:
    name=p["deck"]
    evidence={k+"|"+field:cards for (deck,k,field),cards in card_examples.items() if deck==name}
    for r in card_rows:
        if r["deck"]==name and "Creature" in r["types"].split("; "):
            for t in r["subtypes"].split("; "):
                if t:evidence.setdefault("Leading tribes|"+t,[]).append(r["name"])
    deck_list = [zone+"\n"+"\n".join(f'{r["quantity"]} {r["name"]}' for r in card_rows if r["deck"]==name and r["zone"]==zone) for zone in ("Commander","Deck")]
    payload[name]={**p,"audit":[row for row in audits if row["deck"]==name],"review":route_by_name[name],"name":name,"source":urls.get(name,""),"list":deck_list,"evidence":evidence,"charts":{v:profile_plot(name,v) for v in ["Mana curve","Card types","Leading tribes"]}}
options=lambda names:"".join(f'<option>{label(n)}</option>' for n in names)
page = Path("gameplay.template.html").read_text(encoding="utf-8")
parts={"FIELDSCRIPT":Path("across-decks.js").read_text(encoding="utf-8"),"DETAILSCRIPT":Path("deck-detail.js").read_text(encoding="utf-8"),"VIEWS":"".join(f'<button type="button">{label(v)}</button>' for v in views_matrix),
       "OPTIONS":options(by_name), "GROUPS":options(["All references","Bracket 4","Bracket 3","Bracket 2","Stock precon","Reference means"]),
       "LEGEND":"".join(f'<span><i class="s{i}"></i>{s}</span>' for i,s in enumerate(["Focus deck","Bracket 4 mean","Bracket 3 mean","Bracket 2 mean"])),
       "FIT":profile_plot(seat["deck"],"Table fit"),"TABLES":"".join(make_panel(v,i) for i,v in enumerate(views)),
       "DATA":json.dumps({"defs":role_definitions,"means":read_results("cohort_measures.csv"),"decks":payload,"seat":seat["display"],"matrices":matrices},ensure_ascii=False,separators=(",",":")).replace("<","\\u003c")}
parts.update(PACKPANEL=pack_panel, PACKCSS=Path("companion.css").read_text(encoding="utf-8"), TOOLCSS=Path("compare-tools.css").read_text(encoding="utf-8"), TOOLSCRIPT=Path("compare-tools.js").read_text(encoding="utf-8"))
for key,part in parts.items():page=page.replace("@@"+key+"@@",part)
Path("gameplay.html").write_text(page,encoding="utf-8")
